# AITuber — CogVideoX1.5 Thinking E / T4 RUN-ALL validation
Purpose: one-click Google Colab validation. Input image is embedded in this notebook, so Run all never pauses for manual upload.
CUA rule: Computer Use only starts Run all, then exits as soon as Colab GPU work is visibly running. 2 minutes with no UI progress = stop CUA.

In [ ]:
!nvidia-smi
import torch, time
assert torch.cuda.is_available(), 'GPU runtime is not enabled. In Colab choose Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0), 'capability:', torch.cuda.get_device_capability(0))
print('CUDA:', torch.version.cuda, 'PyTorch:', torch.__version__)
print('STEP 1/5 GPU_READY', time.strftime('%H:%M:%S'))

In [ ]:
%pip install -q -U diffusers transformers accelerate 'optimum-quanto>=0.2.6' imageio-ffmpeg sentencepiece huggingface_hub
print('STEP 2/5 DEPENDENCIES_READY')

In [ ]:
import base64, pathlib, time
INPUT_IMAGE = '/content/thinking_e_768x1344_rgb.png'
_IMAGE_B64 = '...'
pathlib.Path(INPUT_IMAGE).write_bytes(base64.b64decode(_IMAGE_B64))
del _IMAGE_B64
print('Embedded input restored:', INPUT_IMAGE, pathlib.Path(INPUT_IMAGE).stat().st_size, 'bytes')
print('STEP 3/5 INPUT_READY', time.strftime('%H:%M:%S'))

In [ ]:
import gc, time, torch
from diffusers import CogVideoXImageToVideoPipeline
from diffusers.quantizers import PipelineQuantizationConfig

MODEL_ID = 'zai-org/CogVideoX1.5-5B-I2V'
major, minor = torch.cuda.get_device_capability(0)
COMPUTE_DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print('compute dtype:', COMPUTE_DTYPE)
print('Loading model; this is the first long step...', time.strftime('%H:%M:%S'))

quant_config = PipelineQuantizationConfig(
    quant_backend='quanto',
    quant_kwargs={'weights_dtype': 'int8'},
    components_to_quantize=['transformer', 'text_encoder'],
)
pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=COMPUTE_DTYPE,
)
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()
gc.collect(); torch.cuda.empty_cache()
print('STEP 4/5 PIPELINE_READY', time.strftime('%H:%M:%S'))

In [ ]:
import pathlib, time, torch
from diffusers.utils import load_image, export_to_video

PROMPT = (
    'A full-body anime xianxia woman remains in exactly the same thinking pose. '
    'Only subtle natural breathing motion: the upper torso and shoulders gently rise and fall, '
    'with a very small relaxed body sway. Her face, hairstyle, hands, fingers, costume, body proportions '
    'and silhouette remain consistent with the input image. Mouth stays closed. No speaking. '
    'Hair and clothing are almost still. Static camera, fixed framing, no zoom, no pan, no scene change.'
)
NEGATIVE = (
    'extra limbs, extra arms, extra hands, extra fingers, duplicate body parts, deformed hands, '
    'face change, costume change, body shape change, camera movement, zoom, pan, rotation, '
    'walking, large gesture, speaking, mouth movement, background movement, flicker, blur'
)
OUTPUT = '/content/xianxia_thinking_e_cogvideox15_test.mp4'
image = load_image(INPUT_IMAGE)
generator = torch.Generator(device='cuda').manual_seed(42)
print('GENERATION_START', time.strftime('%H:%M:%S'))
frames = pipe(
    image=image,
    prompt=PROMPT,
    negative_prompt=NEGATIVE,
    num_videos_per_prompt=1,
    num_inference_steps=30,
    num_frames=49,
    guidance_scale=6.0,
    generator=generator,
).frames[0]
export_to_video(frames, OUTPUT, fps=16)
assert pathlib.Path(OUTPUT).exists() and pathlib.Path(OUTPUT).stat().st_size > 10000, 'MP4 output missing or too small'
print('STEP 5/5 VIDEO_READY', OUTPUT, pathlib.Path(OUTPUT).stat().st_size, 'bytes', time.strftime('%H:%M:%S'))

In [ ]:
from IPython.display import Video, display
from google.colab import files
display(Video(OUTPUT, embed=True))
print('PASS: MP4 generated. Starting browser download...')
files.download(OUTPUT)